# Video Knowledge Extractor: GPU pipeline
Run top to bottom. Needs: GPU on, Internet on, Secrets `HF_TOKEN` and (for the LLM step) `LLM_API_KEY`, plus the project attached as a dataset.

In [ ]:
!pip install -q faster-whisper pyannote.audio sentence-transformers mediapipe==0.10.14 opencv-python-headless openai python-dotenv
!apt-get -qq install -y ffmpeg > /dev/null
# If pyannote and torch conflict: restart the session and install pyannote.audio first.
# After it works, pin the versions here:  !pip freeze | grep -i -E 'torch|pyannote|whisper|mediapipe'

In [ ]:
import os, sys, glob
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
os.environ['HF_TOKEN'] = s.get_secret('HF_TOKEN')
for key in ['LLM_API_KEY','LLM_BASE_URL','LLM_MODEL','AZURE_OPENAI_ENDPOINT','AZURE_OPENAI_API_KEY','AZURE_OPENAI_DEPLOYMENT']:
    try: os.environ[key] = s.get_secret(key)
    except Exception: pass
os.environ['JOBS_DIR'] = '/kaggle/working/jobs'

In [ ]:
# Point this at the attached project dataset (the folder that contains run_pipeline.py)
hits = glob.glob('/kaggle/input/**/run_pipeline.py', recursive=True)
print(hits)
sys.path.insert(0, os.path.dirname(hits[0]))

In [ ]:
import torch
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# Your demo videos (attached as a dataset)
videos = sorted(glob.glob('/kaggle/input/**/*.mp4', recursive=True))
print(*videos, sep='\n')

In [ ]:
from run_pipeline import process_video

# Start with ONE clean clip and --no-faces style flags to test the basics:
# process_video(videos[0], 'demo1', use_faces=False, use_llm=False, embed=False)

process_video(videos[0], 'demo1')

In [ ]:
# Check the output
import json
r = json.load(open('/kaggle/working/jobs/demo1/result.json'))
print(r['title'], '|', r['topic'][:200])
print([ (s['id'], s['name'], s['on_camera']) for s in r['speakers'] ])
for m in r['messages'][:8]: print(m['speaker'], round(m['start']), m['text'][:80])

In [ ]:
# Zip every finished job for download (Output tab) -> unzip into backend/jobs/ on Azure
!cd /kaggle/working && zip -rq jobs.zip jobs && ls -lh jobs.zip